# Gradient Checkpointing: Trading Compute for Memory

**Historical Context**: Gradient checkpointing (also called activation checkpointing or rematerialization) was introduced by Chen et al. in 2016 as a technique to reduce memory consumption during backpropagation by recomputing activations instead of storing them.

**Key Innovation**: Instead of storing all intermediate activations during the forward pass (O(n) memory for n layers), checkpoint only selected activations and recompute the rest during backward pass. This trades ~30% extra compute for 80%+ memory savings.

**Learning Objectives**:
- Understand activation checkpointing concept
- Learn the memory vs compute trade-off
- Master selective checkpointing strategies
- Implement checkpointing in PyTorch
- Use checkpointing with transformers
- Calculate memory savings
- Know when to use checkpointing

In [ ]:
import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint
import matplotlib.pyplot as plt
import numpy as np
import time
from typing import List, Tuple
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 1. The Activation Memory Problem

**Standard Backpropagation**:
```python
# Forward pass
x1 = layer1(x0)  # Store x1 for backward
x2 = layer2(x1)  # Store x2 for backward
x3 = layer3(x2)  # Store x3 for backward
...
loss = loss_fn(xn)

# Backward pass
# Use stored x1, x2, x3... to compute gradients
```

**Memory Problem**:
- Must store activations from ALL layers
- For sequence length S, batch size B, hidden dim H, layers L:
  - Activation memory = O(S × B × H × L)
- Example: GPT-3 with S=2048, B=8, H=12288, L=96
  - Activations ≈ 19GB per example!

**Gradient Checkpointing Solution**:
```python
# Forward pass
x1 = layer1(x0)  # Store x1 (checkpoint)
x2 = layer2(x1)  # Don't store
x3 = layer3(x2)  # Store x3 (checkpoint)
x4 = layer4(x3)  # Don't store
...

# Backward pass
# Recompute x2 from x1, recompute x4 from x3, etc.
```

In [ ]:
def calculate_activation_memory(
    seq_len: int,
    batch_size: int,
    hidden_dim: int,
    num_layers: int,
    use_checkpointing: bool = False,
    checkpoint_every: int = 2
) -> Tuple[float, int]:
    """
    Calculate activation memory for transformer model.
    
    Args:
        seq_len: Sequence length
        batch_size: Batch size
        hidden_dim: Hidden dimension
        num_layers: Number of layers
        use_checkpointing: Whether to use gradient checkpointing
        checkpoint_every: Checkpoint every N layers
    
    Returns:
        (memory_gb, num_stored_activations)
    """
    # Bytes per activation (FP16)
    bytes_per_activation = 2
    
    # Activations per layer
    activation_size = seq_len * batch_size * hidden_dim * bytes_per_activation
    
    if not use_checkpointing:
        # Store all activations
        total_memory = activation_size * num_layers
        num_stored = num_layers
    else:
        # Store only checkpointed activations
        num_checkpoints = (num_layers + checkpoint_every - 1) // checkpoint_every
        total_memory = activation_size * num_checkpoints
        num_stored = num_checkpoints
    
    memory_gb = total_memory / (1024**3)
    return memory_gb, num_stored

# Example calculations
print("Activation Memory for Different Model Sizes:")
print("="*100)
print(f"{'Model':<20} {'Config':<30} {'No Checkpoint':<20} {'With Checkpoint':<20} {'Savings'}")
print("="*100)

configs = [
    ('GPT-2 Small', 512, 4, 768, 12),
    ('GPT-2 Medium', 512, 4, 1024, 24),
    ('GPT-2 Large', 512, 4, 1280, 36),
    ('GPT-3 6.7B', 2048, 8, 4096, 32),
    ('GPT-3 13B', 2048, 8, 5120, 40),
]

for model_name, seq_len, batch_size, hidden_dim, num_layers in configs:
    mem_no_ckpt, _ = calculate_activation_memory(seq_len, batch_size, hidden_dim, num_layers, False)
    mem_with_ckpt, _ = calculate_activation_memory(seq_len, batch_size, hidden_dim, num_layers, True, checkpoint_every=1)
    savings = (1 - mem_with_ckpt / mem_no_ckpt) * 100
    
    config_str = f"L={num_layers}, H={hidden_dim}"
    print(f"{model_name:<20} {config_str:<30} {mem_no_ckpt:<20.2f} {mem_with_ckpt:<20.2f} {savings:.1f}%")

print("\nNote: This only accounts for activation memory, not parameters or gradients!")

In [ ]:
# Visualize activation storage patterns
def visualize_checkpointing_strategies():
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    num_layers = 12
    strategies = [
        ('No Checkpointing\n(Store all activations)', [1]*num_layers),
        ('Checkpoint Every Layer\n(Store none)', [0]*num_layers),
        ('Checkpoint Every 2 Layers\n(Recommended)', [1, 0] * (num_layers//2)),
        ('Checkpoint Every 4 Layers\n(Less memory saving)', [1, 0, 0, 0] * (num_layers//4)),
    ]
    
    for ax, (title, pattern) in zip(axes, strategies):
        stored = [i for i, p in enumerate(pattern) if p == 1]
        recomputed = [i for i, p in enumerate(pattern) if p == 0]
        
        # Draw layers
        for i in range(num_layers):
            if pattern[i] == 1:
                color = '#4ECDC4'  # Stored (checkpoint)
                label = 'Stored' if i == stored[0] else ''
            else:
                color = '#FF6B6B'  # Recomputed
                label = 'Recomputed' if i == recomputed[0] else ''
            
            ax.barh(0, 1, left=i, color=color, edgecolor='black', 
                   linewidth=2, label=label, alpha=0.8)
            ax.text(i + 0.5, 0, str(i), ha='center', va='center', 
                   fontsize=9, fontweight='bold')
        
        # Add arrows for recomputation
        if len(recomputed) > 0:
            for rc in recomputed:
                # Find previous checkpoint
                prev_ckpt = max([s for s in stored if s < rc] + [-1])
                if prev_ckpt >= 0:
                    ax.annotate('', xy=(rc + 0.5, -0.3), xytext=(prev_ckpt + 0.5, -0.3),
                              arrowprops=dict(arrowstyle='->', color='red', lw=2))
        
        ax.set_xlim(0, num_layers)
        ax.set_ylim(-0.5, 0.5)
        ax.set_yticks([])
        ax.set_xticks([])
        ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
        ax.legend(loc='upper right', fontsize=9)
        
        # Add memory info
        num_stored = sum(pattern)
        memory_fraction = num_stored / num_layers
        ax.text(num_layers/2, -0.4, 
               f'Stored: {num_stored}/{num_layers} ({memory_fraction*100:.0f}% memory)',
               ha='center', fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/checkpointing_strategies.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_checkpointing_strategies()

## 2. Memory vs Compute Trade-off

**The Trade-off**:
- **Without checkpointing**: Store all activations, 1× compute
- **With checkpointing**: Store fewer activations, ~1.3-1.5× compute

**Why is it worth it?**
- Memory is often the bottleneck (can't fit larger batch/sequence)
- 30% extra compute is acceptable for 75%+ memory savings
- Enables training models that wouldn't fit otherwise

**Optimal Strategy**:
- Checkpoint every √n layers for n-layer model (minimizes recomputation)
- In practice: checkpoint every 1-2 layers works well

In [ ]:
def analyze_checkpoint_tradeoff(num_layers: int = 24):
    """
    Analyze memory vs compute trade-off for different checkpointing frequencies.
    """
    checkpoint_frequencies = list(range(1, num_layers + 1))
    memory_fractions = []
    compute_overheads = []
    
    for freq in checkpoint_frequencies:
        # Memory: fraction of activations stored
        num_checkpoints = (num_layers + freq - 1) // freq
        memory_fraction = num_checkpoints / num_layers
        memory_fractions.append(memory_fraction)
        
        # Compute: need to recompute between checkpoints
        # Average recomputation per layer
        avg_recompute = (freq - 1) / 2
        compute_overhead = 1 + avg_recompute / num_layers
        compute_overheads.append(compute_overhead)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Memory savings
    ax = axes[0]
    ax.plot(checkpoint_frequencies, memory_fractions, 'o-', linewidth=2, markersize=8, color='#4ECDC4')
    ax.axhline(y=0.5, color='red', linestyle='--', linewidth=2, alpha=0.5, label='50% memory')
    ax.set_xlabel('Checkpoint Every N Layers', fontsize=12)
    ax.set_ylabel('Memory Fraction (vs no checkpointing)', fontsize=12)
    ax.set_title(f'Memory Usage ({num_layers} layers)', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 1.1)
    
    # Compute overhead
    ax = axes[1]
    ax.plot(checkpoint_frequencies, compute_overheads, 's-', linewidth=2, markersize=8, color='#FF6B6B')
    ax.axhline(y=1.0, color='green', linestyle='--', linewidth=2, alpha=0.5, label='No overhead')
    ax.axhline(y=1.3, color='orange', linestyle='--', linewidth=2, alpha=0.5, label='30% overhead')
    ax.set_xlabel('Checkpoint Every N Layers', fontsize=12)
    ax.set_ylabel('Compute Overhead (vs no checkpointing)', fontsize=12)
    ax.set_title(f'Computational Cost ({num_layers} layers)', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)
    ax.set_ylim(0.9, max(compute_overheads) * 1.1)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/checkpoint_tradeoff.png',
                dpi=150, bbox_inches='tight')
    plt.show()
    
    # Find optimal checkpoint frequency (minimize compute for 50% memory)
    target_memory = 0.5
    optimal_freq = None
    for freq, mem_frac in zip(checkpoint_frequencies, memory_fractions):
        if mem_frac <= target_memory:
            optimal_freq = freq
            break
    
    if optimal_freq:
        idx = checkpoint_frequencies.index(optimal_freq)
        print(f"\nOptimal Strategy for {num_layers} layers:")
        print(f"  Checkpoint every {optimal_freq} layers")
        print(f"  Memory: {memory_fractions[idx]*100:.1f}%")
        print(f"  Compute overhead: {(compute_overheads[idx]-1)*100:.1f}%")
    
    # Theoretical optimum: checkpoint every sqrt(n) layers
    theoretical_freq = int(np.sqrt(num_layers))
    idx_theory = checkpoint_frequencies.index(theoretical_freq)
    print(f"\nTheoretical Optimum (√n = {theoretical_freq}):")
    print(f"  Checkpoint every {theoretical_freq} layers")
    print(f"  Memory: {memory_fractions[idx_theory]*100:.1f}%")
    print(f"  Compute overhead: {(compute_overheads[idx_theory]-1)*100:.1f}%")

analyze_checkpoint_tradeoff(num_layers=24)

## 3. Implementation in PyTorch

PyTorch provides `torch.utils.checkpoint` for gradient checkpointing:

In [ ]:
# Simple example: checkpointing a function
class SimpleLayer(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.linear1 = nn.Linear(dim, dim * 4)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(dim * 4, dim)
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

class ModelWithoutCheckpointing(nn.Module):
    """Model without gradient checkpointing."""
    def __init__(self, num_layers=12, dim=512):
        super().__init__()
        self.layers = nn.ModuleList([SimpleLayer(dim) for _ in range(num_layers)])
    
    def forward(self, x):
        for layer in self.layers:
            x = x + layer(x)  # Residual connection
        return x

class ModelWithCheckpointing(nn.Module):
    """Model with gradient checkpointing."""
    def __init__(self, num_layers=12, dim=512):
        super().__init__()
        self.layers = nn.ModuleList([SimpleLayer(dim) for _ in range(num_layers)])
    
    def forward(self, x):
        for layer in self.layers:
            # Use checkpoint for each layer
            x = x + checkpoint.checkpoint(layer, x, use_reentrant=False)
        return x

# Test both models
batch_size, seq_len, dim = 4, 128, 512
num_layers = 12

x = torch.randn(batch_size, seq_len, dim, device=device, requires_grad=True)

print("Comparing Models:")
print("="*70)

# Without checkpointing
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()

model_no_ckpt = ModelWithoutCheckpointing(num_layers, dim).to(device)
output_no_ckpt = model_no_ckpt(x)
loss = output_no_ckpt.mean()

if device.type == 'cuda':
    torch.cuda.synchronize()
start = time.time()
loss.backward()
if device.type == 'cuda':
    torch.cuda.synchronize()
time_no_ckpt = time.time() - start

if device.type == 'cuda':
    memory_no_ckpt = torch.cuda.max_memory_allocated() / (1024**2)
    print(f"Without Checkpointing:")
    print(f"  Memory: {memory_no_ckpt:.2f} MB")
    print(f"  Backward time: {time_no_ckpt*1000:.2f} ms")

# With checkpointing
if device.type == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

x = torch.randn(batch_size, seq_len, dim, device=device, requires_grad=True)
model_ckpt = ModelWithCheckpointing(num_layers, dim).to(device)
# Copy weights for fair comparison
model_ckpt.load_state_dict(model_no_ckpt.state_dict())

output_ckpt = model_ckpt(x)
loss = output_ckpt.mean()

if device.type == 'cuda':
    torch.cuda.synchronize()
start = time.time()
loss.backward()
if device.type == 'cuda':
    torch.cuda.synchronize()
time_ckpt = time.time() - start

if device.type == 'cuda':
    memory_ckpt = torch.cuda.max_memory_allocated() / (1024**2)
    print(f"\nWith Checkpointing:")
    print(f"  Memory: {memory_ckpt:.2f} MB")
    print(f"  Backward time: {time_ckpt*1000:.2f} ms")
    
    print(f"\nSavings:")
    print(f"  Memory reduction: {(1 - memory_ckpt/memory_no_ckpt)*100:.1f}%")
    print(f"  Compute overhead: {(time_ckpt/time_no_ckpt - 1)*100:.1f}%")
else:
    print("\nRun on GPU to see memory comparison")

print("="*70)

## 4. Selective Checkpointing Strategies

Not all layers benefit equally from checkpointing:

### Strategy 1: Checkpoint Every N Layers
- Simple and effective
- N=1: maximum memory savings
- N=2: good balance

### Strategy 2: Checkpoint Only Attention Layers
- Attention has large activations (seq_len²)
- FFN layers have smaller activations
- Checkpoint attention, skip FFN

### Strategy 3: Checkpoint Expensive Layers
- Checkpoint layers with most activations
- Skip cheap layers (LayerNorm, etc.)

### Strategy 4: Adaptive Checkpointing
- Checkpoint based on available memory
- Dynamic during training

In [ ]:
class TransformerBlockWithSelectiveCheckpointing(nn.Module):
    """Transformer block with selective checkpointing options."""
    def __init__(self, d_model=512, nhead=8, checkpoint_attention=True, checkpoint_ffn=False):
        super().__init__()
        self.checkpoint_attention = checkpoint_attention
        self.checkpoint_ffn = checkpoint_ffn
        
        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Attention block
        if self.checkpoint_attention:
            attn_out = checkpoint.checkpoint(
                lambda x: self.attention(x, x, x)[0],
                x,
                use_reentrant=False
            )
        else:
            attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        
        # FFN block
        if self.checkpoint_ffn:
            ffn_out = checkpoint.checkpoint(self.ffn, x, use_reentrant=False)
        else:
            ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

# Compare different strategies
def compare_checkpoint_strategies():
    strategies = [
        ('No Checkpointing', False, False),
        ('Attention Only', True, False),
        ('FFN Only', False, True),
        ('Both', True, True),
    ]
    
    batch_size, seq_len, d_model = 4, 256, 512
    num_layers = 6
    
    results = []
    
    for strategy_name, ckpt_attn, ckpt_ffn in strategies:
        if device.type != 'cuda':
            continue
            
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
        # Create model
        layers = [TransformerBlockWithSelectiveCheckpointing(
            d_model, 8, ckpt_attn, ckpt_ffn
        ) for _ in range(num_layers)]
        model = nn.Sequential(*layers).to(device)
        
        # Forward + backward
        x = torch.randn(batch_size, seq_len, d_model, device=device, requires_grad=True)
        
        torch.cuda.synchronize()
        start = time.time()
        
        output = model(x)
        loss = output.mean()
        loss.backward()
        
        torch.cuda.synchronize()
        elapsed = time.time() - start
        
        memory_mb = torch.cuda.max_memory_allocated() / (1024**2)
        
        results.append({
            'Strategy': strategy_name,
            'Memory (MB)': memory_mb,
            'Time (ms)': elapsed * 1000,
        })
    
    if results:
        df = pd.DataFrame(results)
        baseline_memory = results[0]['Memory (MB)']
        baseline_time = results[0]['Time (ms)']
        
        df['Memory Savings'] = df['Memory (MB)'].apply(
            lambda x: f"{(1 - x/baseline_memory)*100:.1f}%"
        )
        df['Time Overhead'] = df['Time (ms)'].apply(
            lambda x: f"{(x/baseline_time - 1)*100:.1f}%"
        )
        
        print("\nSelective Checkpointing Strategy Comparison:")
        print("="*100)
        print(df.to_string(index=False))
        print("="*100)
        print("\nRecommendation: Checkpoint attention layers for best memory/speed balance")
    else:
        print("GPU required for memory comparison")

compare_checkpoint_strategies()

## 5. Integration with Hugging Face Transformers

Most Hugging Face models support gradient checkpointing:

In [ ]:
# Example: Using gradient checkpointing with Hugging Face
hf_example = '''
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model
model = AutoModelForCausalLM.from_pretrained(
    "gpt2-large",
    torch_dtype=torch.float16,
    device_map="auto"
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Now train as normal - activations will be checkpointed
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

for batch in dataloader:
    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()  # Checkpointing happens here
    optimizer.step()
    optimizer.zero_grad()

# Disable if needed
model.gradient_checkpointing_disable()

# For fine-grained control:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config.from_pretrained("gpt2-large")
config.use_cache = False  # Required for gradient checkpointing
model = GPT2LMHeadModel(config)
model.gradient_checkpointing_enable()

# With training arguments:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./output",
    gradient_checkpointing=True,  # Enable checkpointing
    per_device_train_batch_size=8,
    fp16=True,
    ...
)

trainer = Trainer(
    model=model,
    args=training_args,
    ...
)
trainer.train()
'''

print("Gradient Checkpointing with Hugging Face Transformers:")
print("="*70)
print(hf_example)
print("\nSupported Models:")
print("-"*70)
print("- All GPT variants (GPT-2, GPT-Neo, GPT-J)")
print("- All LLaMA variants (LLaMA, LLaMA 2, LLaMA 3)")
print("- BERT, RoBERTa, DeBERTa")
print("- T5, BART")
print("- Mistral, Mixtral")
print("- And most other transformer models")

## 6. Memory Savings Calculations

Let's calculate exact memory savings for real models:

In [ ]:
def calculate_total_memory(
    num_parameters: int,
    seq_len: int,
    batch_size: int,
    hidden_dim: int,
    num_layers: int,
    use_checkpointing: bool = False
) -> Dict[str, float]:
    """
    Calculate total memory breakdown including parameters, gradients, and activations.
    """
    # Parameters (mixed precision: FP16 + FP32 master)
    params_memory = (num_parameters * 6) / (1024**3)  # GB
    
    # Gradients (FP16)
    grads_memory = (num_parameters * 2) / (1024**3)
    
    # Optimizer states (Adam: 2 states × FP32)
    optimizer_memory = (num_parameters * 8) / (1024**3)
    
    # Activations
    activation_memory, _ = calculate_activation_memory(
        seq_len, batch_size, hidden_dim, num_layers, use_checkpointing
    )
    
    total_memory = params_memory + grads_memory + optimizer_memory + activation_memory
    
    return {
        'parameters': params_memory,
        'gradients': grads_memory,
        'optimizer': optimizer_memory,
        'activations': activation_memory,
        'total': total_memory,
    }

# Compare models with and without checkpointing
models = [
    ('GPT-2 Large (774M)', 774_000_000, 1280, 36, 1024, 4),
    ('GPT-3 6.7B', 6_700_000_000, 4096, 32, 2048, 8),
    ('GPT-3 13B', 13_000_000_000, 5120, 40, 2048, 8),
]

print("\nMemory Breakdown With vs Without Gradient Checkpointing:")
print("="*120)

for model_name, num_params, hidden_dim, num_layers, seq_len, batch_size in models:
    print(f"\n{model_name}:")
    print("-"*120)
    
    mem_no_ckpt = calculate_total_memory(
        num_params, seq_len, batch_size, hidden_dim, num_layers, False
    )
    mem_ckpt = calculate_total_memory(
        num_params, seq_len, batch_size, hidden_dim, num_layers, True
    )
    
    print(f"{'Component':<20} {'No Checkpoint':<20} {'With Checkpoint':<20} {'Savings'}")
    print("-"*120)
    
    for key in ['parameters', 'gradients', 'optimizer', 'activations', 'total']:
        no_ckpt_val = mem_no_ckpt[key]
        ckpt_val = mem_ckpt[key]
        savings = (1 - ckpt_val / no_ckpt_val) * 100 if no_ckpt_val > 0 else 0
        
        print(f"{key.capitalize():<20} {no_ckpt_val:<20.2f} {ckpt_val:<20.2f} {savings:.1f}%")
    
    print(f"\nFits on A100 40GB? Without: {mem_no_ckpt['total'] <= 40}, With: {mem_ckpt['total'] <= 40}")
    print(f"Fits on A100 80GB? Without: {mem_no_ckpt['total'] <= 80}, With: {mem_ckpt['total'] <= 80}")

print("\n" + "="*120)

In [ ]:
# Visualize memory breakdown
def visualize_memory_breakdown():
    model_name = 'GPT-3 6.7B'
    num_params = 6_700_000_000
    hidden_dim = 4096
    num_layers = 32
    seq_len = 2048
    batch_size = 8
    
    mem_no_ckpt = calculate_total_memory(
        num_params, seq_len, batch_size, hidden_dim, num_layers, False
    )
    mem_ckpt = calculate_total_memory(
        num_params, seq_len, batch_size, hidden_dim, num_layers, True
    )
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    categories = ['Parameters', 'Gradients', 'Optimizer', 'Activations']
    no_ckpt_values = [
        mem_no_ckpt['parameters'],
        mem_no_ckpt['gradients'],
        mem_no_ckpt['optimizer'],
        mem_no_ckpt['activations'],
    ]
    ckpt_values = [
        mem_ckpt['parameters'],
        mem_ckpt['gradients'],
        mem_ckpt['optimizer'],
        mem_ckpt['activations'],
    ]
    
    x = np.arange(len(categories))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, no_ckpt_values, width, label='No Checkpointing', alpha=0.8, color='#FF6B6B')
    bars2 = ax.bar(x + width/2, ckpt_values, width, label='With Checkpointing', alpha=0.8, color='#4ECDC4')
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}GB',
                   ha='center', va='bottom', fontsize=9)
    
    # Add total line
    ax.axhline(y=40, color='red', linestyle='--', linewidth=2, label='A100 40GB', alpha=0.7)
    ax.axhline(y=80, color='orange', linestyle='--', linewidth=2, label='A100 80GB', alpha=0.7)
    
    # Add total annotations
    ax.text(len(categories) - 0.6, mem_no_ckpt['total'] + 2, 
           f"Total: {mem_no_ckpt['total']:.1f}GB",
           fontsize=11, fontweight='bold', color='#FF6B6B')
    ax.text(len(categories) - 0.6, mem_ckpt['total'] + 2,
           f"Total: {mem_ckpt['total']:.1f}GB",
           fontsize=11, fontweight='bold', color='#4ECDC4')
    
    ax.set_ylabel('Memory (GB)', fontsize=12)
    ax.set_title(f'Memory Breakdown: {model_name}\n(Seq={seq_len}, Batch={batch_size})', 
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=11)
    ax.legend(fontsize=10, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/notebooks/checkpoint_memory_breakdown.png',
                dpi=150, bbox_inches='tight')
    plt.show()

visualize_memory_breakdown()

## 7. When to Use Gradient Checkpointing

### ✅ Use Checkpointing When:

1. **Memory Constrained**:
   - Model doesn't fit in GPU memory
   - Want larger batch sizes
   - Want longer sequences

2. **Large Models**:
   - >1B parameters
   - Deep architectures (>24 layers)
   - Large hidden dimensions

3. **Long Sequences**:
   - Sequence length >1024
   - Activations dominate memory

4. **Training (not inference)**:
   - Only beneficial during training
   - No activation storage needed for inference

### ❌ Don't Use Checkpointing When:

1. **Memory is Plenty**:
   - Model fits comfortably
   - Small models (<500M parameters)

2. **Speed is Critical**:
   - Tight training deadlines
   - Already slow training
   - 30% slowdown is unacceptable

3. **Inference**:
   - No gradients needed
   - No backward pass
   - Use model.eval() instead

4. **Short Sequences**:
   - Sequence length <256
   - Activation memory is small

In [ ]:
# Decision helper function
def should_use_checkpointing(
    num_parameters: int,
    seq_len: int,
    batch_size: int,
    hidden_dim: int,
    num_layers: int,
    gpu_memory_gb: float = 40,
    speed_priority: bool = False
) -> Tuple[bool, str]:
    """
    Decide whether to use gradient checkpointing.
    
    Returns:
        (should_checkpoint, reasoning)
    """
    mem_no_ckpt = calculate_total_memory(
        num_parameters, seq_len, batch_size, hidden_dim, num_layers, False
    )
    mem_ckpt = calculate_total_memory(
        num_parameters, seq_len, batch_size, hidden_dim, num_layers, True
    )
    
    # Check if it fits without checkpointing
    fits_without = mem_no_ckpt['total'] <= gpu_memory_gb * 0.9  # 90% threshold
    fits_with = mem_ckpt['total'] <= gpu_memory_gb * 0.9
    
    if not fits_without and not fits_with:
        return True, (f"Required even with checkpointing ({mem_ckpt['total']:.1f}GB > {gpu_memory_gb}GB). "
                     "Consider reducing batch size or sequence length.")
    
    if not fits_without and fits_with:
        return True, (f"Necessary to fit in memory ({mem_no_ckpt['total']:.1f}GB without vs "
                     f"{mem_ckpt['total']:.1f}GB with checkpointing).")
    
    if fits_without:
        if speed_priority:
            return False, (f"Model fits without checkpointing ({mem_no_ckpt['total']:.1f}GB) and "
                          "speed is priority. Skip checkpointing.")
        else:
            # Check if we can benefit from larger batch
            memory_saved = mem_no_ckpt['total'] - mem_ckpt['total']
            if memory_saved > 10:  # >10GB savings
                return True, (f"Can save {memory_saved:.1f}GB. Use savings for larger batch size "
                             "or longer sequences.")
            else:
                return False, (f"Small memory savings ({memory_saved:.1f}GB). "
                              "Skip checkpointing for better speed.")
    
    return False, "Should not reach here"

# Test decision helper
test_cases = [
    ('Small model, short seq', 124_000_000, 512, 4, 768, 12, 40, False),
    ('Large model, long seq', 6_700_000_000, 2048, 8, 4096, 32, 40, False),
    ('Huge model', 13_000_000_000, 2048, 8, 5120, 40, 40, False),
    ('Speed priority', 1_300_000_000, 1024, 8, 2048, 24, 80, True),
]

print("\nGradient Checkpointing Recommendations:")
print("="*120)

for name, num_params, seq_len, batch_size, hidden_dim, num_layers, gpu_mem, speed_priority in test_cases:
    should_ckpt, reason = should_use_checkpointing(
        num_params, seq_len, batch_size, hidden_dim, num_layers, gpu_mem, speed_priority
    )
    
    print(f"\n{name}:")
    print(f"  Parameters: {num_params/1e9:.1f}B, Layers: {num_layers}, Seq: {seq_len}, Batch: {batch_size}")
    print(f"  Recommendation: {'✅ USE checkpointing' if should_ckpt else '❌ SKIP checkpointing'}")
    print(f"  Reason: {reason}")
    print("-"*120)

## Summary

**Key Takeaways**:

1. **Problem**: Storing activations for backprop uses O(layers × batch × seq_len × hidden_dim) memory

2. **Solution**: Gradient checkpointing saves activations selectively and recomputes during backward pass

3. **Trade-off**:
   - Memory: 50-90% reduction (depending on checkpointing frequency)
   - Compute: 30-50% overhead (recomputation cost)

4. **Strategies**:
   - Checkpoint every layer: Maximum savings
   - Checkpoint every 2 layers: Good balance
   - Selective: Checkpoint attention, skip FFN

5. **Implementation**:
   - PyTorch: `torch.utils.checkpoint.checkpoint()`
   - Hugging Face: `model.gradient_checkpointing_enable()`
   - Automatic in most frameworks

6. **When to Use**:
   - ✅ Large models (>1B params)
   - ✅ Long sequences (>1024 tokens)
   - ✅ Memory constrained
   - ❌ Small models
   - ❌ Speed is critical
   - ❌ Inference

7. **Typical Results**:
   - GPT-2 Large: 70% memory reduction, 35% slower
   - GPT-3 6.7B: 85% memory reduction, 40% slower
   - Enables training 2-3x larger batch sizes

**Papers**:
- Original Paper: [Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174) (Chen et al., 2016)
- PyTorch Docs: https://pytorch.org/docs/stable/checkpoint.html